# Stress Prediction — gender 제거 v34(tree+pair) + SVR 앙상블

- v34(tree+pair, gender 제거) OOF 예측 생성 — 오늘 확정된 최선(CV 0.147650)
- SVR(gender 제거, 내부 grid search로 C/epsilon 튜닝) OOF 예측 생성
- 둘을 **MAE를 직접 최소화하는 가중치**로 블렌드 (RidgeCV는 MSE 최적화라 이전에 실패했음)

⚠️ SVR은 오늘 단독으로 CV MAE 0.19~0.22였습니다. gender 제거로 그 격차가 메워지진
않겠지만, v34에 아주 작은 비중으로 섞였을 때 미세하게라도 도움되는지는 별개 문제라 확인합니다.


In [1]:

from __future__ import annotations

import os
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold

TARGET = "stress_score"
ID_COLUMN = "ID"
RANDOM_SEED = 42
N_ESTIMATORS = 1200
TREE_QUANTILE = 0.54
PAIR_QUANTILE = 0.48
PAIR_WEIGHT = 0.28  # v34 tree+pair 내부 블렌드 (gender 제거 버전에서도 이 값 유지, 이미 검증됨)

DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project/8_6")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COPY_COUNTS = {
    "mean_working": 1, "bmi": 4, "cholesterol": 2, "height": 2,
    "glucose": 3, "weight": 2, "cholesterol_glucose_ratio": 2, "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working", "bmi", "cholesterol", "height",
    "glucose", "weight", "cholesterol_glucose_ratio", "bone_density",
]


def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = result["cholesterol"] / (result["glucose"] + 1e-6)
    result["missing_count"] = result.isna().sum(axis=1)
    return result


def add_feature_copies(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for feature, n_copies in FEATURE_COPY_COUNTS.items():
        for i in range(1, n_copies + 1):
            result[f"{feature}__copy{i}"] = result[feature]
    return result


def make_preprocessor(frame: pd.DataFrame) -> ColumnTransformer:
    categorical = frame.select_dtypes(include=["object", "category", "string"]).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer([
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical),
        ("numerical", SimpleImputer(strategy="median", add_indicator=True), numerical),
    ])


def v34_prediction_no_gender(train_features, val_features, target) -> np.ndarray:
    """gender 제거된 tree + 원본(gender 포함) pair 블렌드 -> 오늘 확정된 candidate B."""
    train_no_gender = train_features.drop(columns=["gender"])
    val_no_gender = val_features.drop(columns=["gender"])
    weighted_train = add_feature_copies(train_no_gender)
    weighted_val = add_feature_copies(val_no_gender)
    preprocessor = make_preprocessor(weighted_train)
    train_matrix = preprocessor.fit_transform(weighted_train)
    val_matrix = preprocessor.transform(weighted_val)
    model = ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS, min_samples_leaf=1, max_features=1,
        criterion="squared_error", random_state=RANDOM_SEED, n_jobs=-1,
    )
    model.fit(train_matrix, target)
    tree_predictions = np.column_stack([t.predict(val_matrix) for t in model.estimators_])
    tree_pred = np.quantile(tree_predictions, TREE_QUANTILE, axis=1)

    train_rank, val_rank = {}, {}
    for column in PAIR_COLUMNS:
        median = np.nanmedian(train_features[column].to_numpy(dtype=float))
        tr = np.nan_to_num(train_features[column].to_numpy(dtype=float), nan=median)
        va = np.nan_to_num(val_features[column].to_numpy(dtype=float), nan=median)
        sorted_tr = np.sort(tr)
        train_rank[column] = np.searchsorted(sorted_tr, tr, side="right") / len(tr)
        val_rank[column] = np.searchsorted(sorted_tr, va, side="right") / len(tr)
    target_values = target.to_numpy(dtype=float)
    neighbor_targets = []
    for first, second in combinations(PAIR_COLUMNS, 2):
        distance = (
            np.abs(val_rank[first][:, None] - train_rank[first][None, :])
            + np.abs(val_rank[second][:, None] - train_rank[second][None, :])
        )
        nearest_idx = np.argmin(distance, axis=1)
        neighbor_targets.append(target_values[nearest_idx])
    pair_pred = np.quantile(np.column_stack(neighbor_targets), PAIR_QUANTILE, axis=1)

    return np.clip(np.round((1 - PAIR_WEIGHT) * tree_pred + PAIR_WEIGHT * pair_pred, 2), 0.0, 1.0)


def svr_prediction_no_gender(train_features, val_features, target) -> np.ndarray:
    train_no_gender = train_features.drop(columns=["gender"])
    val_no_gender = val_features.drop(columns=["gender"])
    preprocessor = make_preprocessor(train_no_gender)
    train_matrix = preprocessor.fit_transform(train_no_gender)
    val_matrix = preprocessor.transform(val_no_gender)
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_matrix)
    val_scaled = scaler.transform(val_matrix)

    param_grid = [(c, e) for c in (1.0, 10.0, 50.0) for e in (0.005, 0.01, 0.02)]
    inner_cv = KFold(3, shuffle=True, random_state=RANDOM_SEED)
    y_arr = target.to_numpy(dtype=float)
    best_score, best_params = np.inf, (10.0, 0.01)
    for c, e in param_grid:
        scores = []
        for inner_tr, inner_va in inner_cv.split(train_scaled):
            m = SVR(kernel="rbf", C=c, epsilon=e, gamma="scale")
            m.fit(train_scaled[inner_tr], y_arr[inner_tr])
            scores.append(mean_absolute_error(y_arr[inner_va], m.predict(train_scaled[inner_va])))
        mean_score = float(np.mean(scores))
        if mean_score < best_score:
            best_score, best_params = mean_score, (c, e)

    final_model = SVR(kernel="rbf", C=best_params[0], epsilon=best_params[1], gamma="scale")
    final_model.fit(train_scaled, target)
    return final_model.predict(val_scaled)


## 5-Fold CV — v34(gender제거) 단독, SVR(gender제거) 단독, 앙상블 비교

In [2]:

train = pd.read_csv(DATA_DIR / "train.csv")
raw_X = add_features(train.drop(columns=[TARGET, ID_COLUMN])).reset_index(drop=True)
y = train[TARGET].reset_index(drop=True)

cv = KFold(5, shuffle=True, random_state=RANDOM_SEED)
oof_v34 = np.zeros(len(y))
oof_svr = np.zeros(len(y))

for fold, (train_idx, val_idx) in enumerate(cv.split(raw_X), start=1):
    train_feat = raw_X.iloc[train_idx].reset_index(drop=True)
    val_feat = raw_X.iloc[val_idx].reset_index(drop=True)
    y_train = y.iloc[train_idx].reset_index(drop=True)

    oof_v34[val_idx] = v34_prediction_no_gender(train_feat, val_feat, y_train)
    oof_svr[val_idx] = svr_prediction_no_gender(train_feat, val_feat, y_train)
    print(f"Fold {fold} done")

mae_v34 = mean_absolute_error(y, oof_v34)
mae_svr = mean_absolute_error(y, oof_svr)
print()
print(f"v34(gender 제거) 단독 CV MAE: {mae_v34:.6f}")
print(f"SVR(gender 제거) 단독 CV MAE: {mae_svr:.6f}")

# MAE 직접 최소화하는 앙상블 가중치 탐색
best_w, best_mae = 1.0, mae_v34
for w in np.arange(0.0, 0.31, 0.02):  # SVR 비중은 최대 30%까지만 (이미 훨씬 나쁜 모델이라)
    blended = np.clip(np.round((1 - w) * oof_v34 + w * oof_svr, 2), 0.0, 1.0)
    mae = mean_absolute_error(y, blended)
    if mae < best_mae:
        best_mae, best_w = mae, w

print()
print(f"최적 SVR 비중: {best_w:.2f}, 앙상블 CV MAE: {best_mae:.6f}")
print(f"(v34 단독 대비 개선: {mae_v34 - best_mae:+.6f})")


Fold 1 done
Fold 2 done
Fold 3 done
Fold 4 done
Fold 5 done

v34(gender 제거) 단독 CV MAE: 0.147787
SVR(gender 제거) 단독 CV MAE: 0.210771

최적 SVR 비중: 1.00, 앙상블 CV MAE: 0.147787
(v34 단독 대비 개선: +0.000000)


## 최종 제출 파일 생성 (앙상블이 v34 단독보다 나을 때만)

In [3]:

BEST_SVR_WEIGHT = 0.0  # 위 결과의 best_w로 수정 (0이면 v34 단독 = candidate B와 동일)

test = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
raw_X_test = add_features(test.drop(columns=[ID_COLUMN])).reset_index(drop=True)

final_v34 = v34_prediction_no_gender(raw_X, raw_X_test, y)
if BEST_SVR_WEIGHT > 0:
    final_svr = svr_prediction_no_gender(raw_X, raw_X_test, y)
    final_prediction = np.clip(np.round((1 - BEST_SVR_WEIGHT) * final_v34 + BEST_SVR_WEIGHT * final_svr, 2), 0.0, 1.0)
else:
    final_prediction = final_v34

output = submission.copy()
output[TARGET] = final_prediction
submit_path = OUTPUT_DIR / "submit_v34nogender_svr_ensemble_0806.csv"
output.to_csv(submit_path, index=False, encoding="utf-8")

print("Saved:", submit_path)
print("Prediction range:", final_prediction.min(), final_prediction.max())
display(output.head(10))


Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_v34nogender_svr_ensemble_0806.csv
Prediction range: 0.0 1.0


,ID,stress_score
0,TEST_0000,0.62
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.51
4,TEST_0004,0.53
5,TEST_0005,0.47
6,TEST_0006,0.78
7,TEST_0007,0.48
8,TEST_0008,0.57
9,TEST_0009,0.88
